- Entradas: "PwmD", "PwmE", "sPwm", "dPwm", "cos(theta_pred)"
- Saida: X
- Loss = L_d + L_p 


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import tensorflow as tf
import matplotlib.pyplot as plt
import joblib
import os
TITLES = [
    "ZZx1",  # Train
    "ZZx2",  # Val
    "ZZxReto", # Test
]
PREDICTORS = ["e_a_d", "e_a_e", "Se_a", "De_a", "costheta"]   
TARGET_INT = ["x"]  
TARGET = ["dx"]
     
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET)   
PLOT = False
TIME_STEPS = 6
TS = 0.07

In [2]:
Datasets = []
NormDatasets = []

for title in TITLES:
    df = pd.read_excel("./../Theta/Data/Datasets.xlsx", sheet_name=title)
    df_theta = pd.read_excel("./../Theta/resultados_comparacao.xlsx", sheet_name=title)

    df["theta_pred"] = df_theta["theta_pred_model_arch77_r0.01_Ld0.5_Lp0.5_seed686"]

    df["costheta"] = np.cos(df["theta_pred"])
    df = df.dropna().reset_index(drop=True)

    Datasets.append(df)

In [3]:

NormDatasets = []

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

Train = Train1

for i in range(len(Datasets) - 1):
    CurrentTestDataset = Datasets[i + 1].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)

Val =NormDatasets[1]

In [4]:
os.makedirs("./scalers", exist_ok=True)
os.makedirs("./Data", exist_ok=True)

with pd.ExcelWriter("./Data/NormDatasets.xlsx", engine="openpyxl") as writer_norm:
    for title, normDataset in zip(TITLES, NormDatasets):
        normDataset.to_excel(writer_norm, sheet_name=title[:31], index=False)

with pd.ExcelWriter("./Data/Datasets.xlsx", engine="openpyxl") as writer:
    for title, Dataset in zip(TITLES, Datasets):
        Dataset.to_excel(writer, sheet_name=title[:31], index=False)
        
joblib.dump(SCALER, "./scalers/scaler.pkl")
joblib.dump(OUT_SCALER, "./scalers/out_scaler.pkl")

mean_tf = tf.constant(OUT_SCALER.mean_[0], dtype=tf.float32)
std_tf  = tf.constant(OUT_SCALER.scale_[0], dtype=tf.float32)        

In [5]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

x_train, y_train = CreateSequences(Train[PREDICTORS], Train[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences(Val[PREDICTORS], Val[TARGET], TIME_STEPS)
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(y_val)}")

Dimensão da entrada: (1024, 6, 5)
Dimensão da saida: (1024, 1)
Dimensão da entrada: (1382, 6, 5)
Dimensão da saida: (1382, 1)


In [6]:
phi_d_train =  Train["phi_d"].values[:len(x_train)]
phi_e_train =  Train["phi_e"].values[:len(x_train)]
theta_train = Train["theta"].values[:len(x_train)]

print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada fisica : {np.shape(phi_d_train)}")
print(f"Dimensão da entrada fisica: {np.shape(phi_e_train)}")

Dimensão da entrada: (1024, 6, 5)
Dimensão da saida: (1024, 1)
Dimensão da entrada fisica : (1024,)
Dimensão da entrada fisica: (1024,)


$$ \dot{\theta} = \frac{R}{2L} (\phi_d - \phi_e) $$
$$ \dot{x} = \frac{R}{2} (\phi_d + \phi_e) (\cos(\theta))$$
$$ \dot{y} = \frac{R}{2} (\phi_d + \phi_e) (\sin(\theta)) $$

In [7]:
R = tf.constant(0.0328, dtype=tf.float32)
L = tf.constant(0.0615, dtype=tf.float32)
dt = tf.constant(TS, dtype=tf.float32)

def CinematicModel(phi_d, phi_e, theta):
    dx_cin = (R / 2) * tf.cos(theta) * (phi_d + phi_e)
    return [dx_cin]

In [8]:

def NumericalIntegration(dataset, dq):

    q = [None] * OUTPUT_SIZE

    init_vals = np.array([
        dataset[name].iloc[0] for name in TARGET_INT
    ])

    for j in range(OUTPUT_SIZE):
        q[j] = init_vals[j] + np.cumsum(dq[j] * TS)

    return q

def GetCin(dataset): 
    dq = CinematicModel(tf.convert_to_tensor(dataset["phi_d"].values, dtype=tf.float32),
                        tf.convert_to_tensor(dataset["phi_e"].values, dtype=tf.float32), 
                        tf.convert_to_tensor(dataset["theta"].values, dtype=tf.float32))
    q = NumericalIntegration(dataset, dq)
    return np.vstack(dq).T, np.vstack(q).T

In [9]:
def BuildRNN(architecture, initializer, regularizer):

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(TIME_STEPS, INPUT_SIZE)))

    for i, units in enumerate(architecture):

        return_sequences = (i < len(architecture) - 1)

        model.add(
            tf.keras.layers.SimpleRNN(
                units,
                activation='tanh',
                return_sequences=return_sequences,
                kernel_initializer=initializer,
                kernel_regularizer=regularizer,
                recurrent_regularizer=regularizer,
                bias_regularizer=regularizer
            )
        )

    model.add(
        tf.keras.layers.Dense(
            OUTPUT_SIZE,
            activation="linear",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer,
            bias_regularizer=regularizer
        )
    )

    return model

In [10]:
@tf.function
def train_step(model, optimizer, x, dy, phi_d, phi_e, theta, Ld, Lp):
    with tf.GradientTape() as tape:

        dy_pred = model(x, training=True)
        
        # loss dos dados
        data_loss = tf.reduce_mean(tf.square(dy_pred - dy))
        
        # termo físico
        physics = tf.stack(CinematicModel(phi_d, phi_e, theta), axis=1)   
             
        # normalização correta
        physics_norm = (physics - mean_tf) / std_tf

        physics_loss = tf.reduce_mean(tf.square(dy_pred - physics_norm))

        loss = Ld * data_loss +  Lp * physics_loss 

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss

In [11]:
phi_d_train = tf.convert_to_tensor(phi_d_train, dtype=tf.float32)
phi_e_train = tf.convert_to_tensor(phi_e_train, dtype=tf.float32)
theta_train = tf.convert_to_tensor(theta_train, dtype=tf.float32)

x_train_tf = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train, dtype=tf.float32)

x_val_tf = tf.convert_to_tensor(x_val, dtype=tf.float32)
y_val_tf = tf.convert_to_tensor(y_val, dtype=tf.float32)

In [12]:
def EarlyStopping(model, best_loss, counter, best_weights, min_delta=1e-3):
    val_pred = model(x_val_tf, training=False)
    val_loss = tf.reduce_mean(tf.square(val_pred - y_val_tf))
    
    if val_loss < (best_loss - min_delta):
        best_loss = val_loss
        counter = 0
        best_weights = model.get_weights()

    else:
        counter += 1

    return best_loss, counter, best_weights, val_loss

def TrainPINN(model, optimizer, Ld, Lp, patience=200, best_loss=np.inf):
    counter = 0
    best_weights = model.get_weights()

    for epoch in range(20000):

        loss = train_step(model, optimizer, x_train_tf, y_train_tf,
                          phi_d_train, phi_e_train, theta_train, Ld, Lp)
        
        best_loss, counter, best_weights, val_loss =  EarlyStopping(model, best_loss, counter, best_weights)
        
        if counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            model.set_weights(best_weights)
            break

        if epoch % 100 == 0:
            print(f"Epoch {epoch} | Train Loss {loss.numpy():.6f} | Val Loss {val_loss.numpy():.6f}")

In [13]:
def PlotOut(ax, title, target_name, y_true, y_pred, y_cin):
    time = (np.arange(len(y_pred)).astype(float) * 0.07).round(5)

    ax.plot(time, y_true, '-', linewidth=1.5, label='Amostras Reais')
    ax.plot(time, y_pred, '--', linewidth=1.5, label='Valores preditos')
    ax.plot(time, y_cin, ':', linewidth=2, label='Modelo Cinemático')

    ax.set_title(f'{title} - {target_name}')
    ax.set_xlabel('Tempo [s]')
    ax.set_ylabel(target_name)
    ax.legend()
    ax.grid(True)


def EvalModel(model):
    from sklearn.metrics import r2_score, mean_squared_error

    n_targets = len(TARGET)
    n_datasets = len(Datasets)

    if PLOT:
        fig, axs = plt.subplots(
            n_datasets,
            2 * n_targets,
            figsize=(6 * 2 * n_targets, 4 * n_datasets)
        )
        axs = np.atleast_2d(axs)

    metrics = {name: {} for name in TARGET_INT}

    for i, NormDataset in enumerate(NormDatasets):

        x = NormDataset[PREDICTORS]
        y = Datasets[i][TARGET_INT]
        dy_true = Datasets[i][TARGET].values

        x, y = CreateSequences(x, y, TIME_STEPS)

        # alinhar derivada
        dy_true = dy_true[TIME_STEPS:]

        pred = model(tf.convert_to_tensor(x, dtype=tf.float32)).numpy()
        dy_pred = OUT_SCALER.inverse_transform(pred)

        y_true = y.copy()
        y_pred = np.zeros_like(dy_pred)

        dy_cin, y_cin = GetCin(Datasets[i])
        y_cin = y_cin[:y_true.shape[0]]
        dy_cin = dy_cin[:dy_pred.shape[0]]

        init_vals = np.array([Datasets[i][name].iloc[0] for name in TARGET_INT])

        for j in range(n_targets):
            y_pred[:, j] = init_vals[j] + np.cumsum(dy_pred[:, j] * TS)

        for j, name in enumerate(TARGET_INT):

            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = r2_score(dy_true[:, j], dy_pred[:, j])

            key_r2 = f"R2_{TITLES[i]}"
            key_mse = f"MSE_{TITLES[i]}"

            metrics[name].setdefault(key_r2, []).append(r2)
            metrics[name].setdefault(key_mse, []).append(mse)

            print(f"{name} | {TITLES[i]} -> R² = {r2:.4f}, R² diff = {mse:.4f}")

            if PLOT:
                ax_y = axs[i, j]
                PlotOut(ax_y, TITLES[i], name,
                        y_true[:, j], y_pred[:, j], y_cin[:, j])

                ax_dy = axs[i, j + n_targets]
                PlotOut(ax_dy, TITLES[i], f"d{name}",
                        dy_true[:, j], dy_pred[:, j], dy_cin[:, j])

    if PLOT:
        plt.tight_layout()

    return metrics

In [14]:
def to_scalar(x):
    return float(x[0]) if isinstance(x, list) else float(x)

In [15]:
def UpdateRow(metrics, arch, Ld, Lp, r, seed, excel_file):

    model_name = f"model_arch{'-'.join(map(str, arch))}_r{r}_Ld{Ld}_Lp{Lp}_seed{seed}"

    row = {
        "model": model_name,
        "Neurons": arch,
        "Ld": Ld,
        "Lp": Lp,
        "reg": r,
        "seed": seed,
    }

    for name in TARGET_INT:
        entry = {}
        for title in TITLES:
            safe_title = title.replace("-", "_")  
            entry[f"R2_{safe_title}_{name}"] = to_scalar(metrics[name][f"R2_{title}"])
            entry[f"MSE_{safe_title}_{name}"] = to_scalar(metrics[name][f"MSE_{title}"])
        row.update(entry)
        df = pd.DataFrame([row])

    try:
        old = pd.read_excel(excel_file)
        new_df = pd.concat([old, df], ignore_index=True)
        new_df.to_excel(excel_file, index=False)
    except FileNotFoundError:
        df.to_excel(excel_file, index=False)

    print(f"Modelo {arch} | Ld={Ld} Lp={Lp} r={r} seed={seed} salvo.")


In [16]:
def ExportModel(model, model_name):

    #os.makedirs("weights", exist_ok=True)
    os.makedirs("models", exist_ok=True)

    model_path = f"models/{model_name}.keras"

    #model.save_weights(weights_path)
    model.save(model_path)

    print(f"Modelo salvo em:\n{model_path}")

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import initializers
from itertools import product
import numpy as np
import tensorflow as tf

N_MODELS = 5

seeds = np.random.choice(range(1, 10000), size=N_MODELS, replace=False)

architectures = [[n] for n in range(70, 101)]


# FirstLayers = [
#     [27], [30], [32], [35], [36], [37], [40],
#     [42], [44], [46], [49], [50], [52], [75],
# ]

# architectures = [
#     [fl[0], sl]
#     for fl in FirstLayers
#     for sl in range(
#         max(1, int(0.3 * fl[0])),
#         int(0.7 * fl[0])
#     )
# ]

# 🔹 retomar a partir do último modelo salvo
# last_saved = [27, 12]

# if last_saved in architectures:
#     idx = architectures.index(last_saved)
#     architectures = architectures[idx + 1:]  # começa DEPOIS do último salvo
#     print(f"Retomando a partir de {last_saved} (índice {idx}). Restam {len(architectures)} arquiteturas.")
# else:
#     print(f"⚠️ {last_saved} não encontrado na lista — rodando tudo do zero.")

print(len(architectures))
print(architectures)

Ld_Lp = [[0.5, 0.5], [0.3, 0.7], [0.7, 0.3]]
r_values = [0.01, 0.9]

results = {}

# produto cartesiano de todos hiperparâmetros
for arch, (Ld, Lp), r in product(architectures, Ld_Lp, r_values):

    for i, s in enumerate(seeds):

        tf.keras.backend.clear_session()

        init = initializers.RandomNormal(seed=int(s))
        reg = tf.keras.regularizers.l2(r)
        model = BuildRNN(arch, init, reg)
        model.build((None, TIME_STEPS, INPUT_SIZE))

        opt = Adam(learning_rate=0.001)
        opt.build(model.trainable_variables)

        TrainPINN(
            model,
            Ld=Ld,
            Lp=Lp,
            optimizer=opt
        )
        model_name = f"model_arch{'-'.join(map(str, arch))}_r{r}_Ld{Ld}_Lp{Lp}_seed{s}"
        ExportModel(model, model_name=model_name)
        metrics = EvalModel(model)
        UpdateRow(metrics, arch, Ld, Lp, r, s, excel_file="resultados-1l.xlsx")

31
[[70], [71], [72], [73], [74], [75], [76], [77], [78], [79], [80], [81], [82], [83], [84], [85], [86], [87], [88], [89], [90], [91], [92], [93], [94], [95], [96], [97], [98], [99], [100]]
Epoch 0 | Train Loss 0.859793 | Val Loss 0.294128
Epoch 100 | Train Loss 0.022002 | Val Loss 0.172810
Epoch 200 | Train Loss 0.018831 | Val Loss 0.251589
Early stopping at epoch 242
Modelo salvo em:
models/model_arch70_r0.01_Ld0.5_Lp0.5_seed1771.keras
x | ZZx1 -> R² = 0.9582, R² diff = 0.9706
x | ZZx2 -> R² = 0.6937, R² diff = 0.7855
x | ZZxReto -> R² = 0.8685, R² diff = 0.9614
Modelo [70] | Ld=0.5 Lp=0.5 r=0.01 seed=1771 salvo.
Epoch 0 | Train Loss 0.676626 | Val Loss 0.240729
Epoch 100 | Train Loss 0.024901 | Val Loss 0.119569
Epoch 200 | Train Loss 0.019987 | Val Loss 0.124322
Early stopping at epoch 222
Modelo salvo em:
models/model_arch70_r0.01_Ld0.5_Lp0.5_seed3379.keras
x | ZZx1 -> R² = 0.9177, R² diff = 0.9467
x | ZZx2 -> R² = 0.8270, R² diff = 0.8322
x | ZZxReto -> R² = 0.7812, R² diff = 0.